In [ ]:
from fastai.vision.all import *

In [ ]:
# Dataset location
path = Path(r'C:\Users\vmhri\Desktop\Trash_Dataset')

#Extracts images from dataset
files = get_image_files(path)
print(f"Total images (Train + Valid): {len(files)}")

In [ ]:
#For second training run (448px)

waste_block_448 = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=GrandparentSplitter(train_name='Train', valid_name='Test'),
    get_y=parent_label,
    item_tfms=Resize(512), 
    batch_tfms=[*aug_transforms(size=448, min_scale=0.75), 
                Normalize.from_stats(*imagenet_stats)]
)

dls_448 = waste_block_448.dataloaders(
    path,
    bs=4,               
    num_workers=4, 
    pin_memory=True
)

In [ ]:
# Load your saved model
learn = load_learner('waste_model_448.pkl') 
learn.dls = dls_448
# Generate the matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
#For first training run (224px)

waste_block_224 = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=GrandparentSplitter(train_name='Train', valid_name='Test'),
    get_y=parent_label,
    # 1. PRESIZING: CPU does one quick large crop
    item_tfms=Resize(460), 
    # 2. GPU AUGMENTATION: GPU handles the final 224 resize and flips in parallel
    batch_tfms=[*aug_transforms(size=224, min_scale=0.75), 
                Normalize.from_stats(*imagenet_stats)]
)
dls_224 = waste_block_224.dataloaders(
    path,
    bs=6,
    num_workers=4,      # Uses multiple CPU cores to prep images
    pin_memory=True,     # Creates a "fast lane" from RAM to GPU
    prefetch_factor=2    # Preps the next batch while the current one is training
)

In [ ]:
dls_224.show_batch()

In [ ]:
#For second training run (448px)
  
learn = load_learner('waste_model.pkl')
learn.dls = dls_448
learn.to_fp16()

In [ ]:
#For first training run (224px)

learn = vision_learner(dls_224, 'convnext_tiny', metrics=error_rate).to_fp16() #fp16() says python to uses 16bit math instead of 32bit math(increase speed)
lrs = learn.lr_find(suggest_funcs=(valley))

In [ ]:
# Use this to prevent the progress bar from ever starting
import fastprogress
fastprogress.fastprogress.printing = lambda: True

from fastai.test_utils import *
#  the fine-tuning
with learn.no_bar():
    learn.fine_tune(3, base_lr=1e-4) #For first training run we use lr provided by valley algorithm

In [ ]:
#Sample images to test
alum = 'samples/alum_1_pred.jpg'
e_waste = 'samples/e_waste_3_pred.jpg'
plastic_hdpe = 'samples/plastic_HDPE_1_pred.jpg'
plastic_pet = 'samples/plastic_PET_1_pred.jpg'

pred, pred_idx, probs = learn.predict(plastic_hdpe)

print(f"Prediction: {pred}")
print(f"Confidence: {probs[pred_idx]:.4f}")

In [ ]:
learn.export('waste_model_448.pkl')